In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/Users/pramukhvenkateshkoushik/Desktop/Data_cap")
YEARS= [2025,2026] 
QUARTER_DIRS = [BASE/ f"{y}q{q}" for y in YEARS for q in range(1, 5) if (BASE / f"{y}q{q}").exists()] 
FORMS = ["10-K", "10-Q", "10-K/A", "10-Q/A"]   
print("Quarter directories:", QUARTER_DIRS)

# i used help of AI and Pawan's notebook to find out the major tags for the features we need. I have also added some of the older tags that were used in the past
# for the same features.
FEATURE_TAGS = {
    "total_assets":        ["Assets"],
    "current_assets":      ["AssetsCurrent"],
    "total_liabilities":   ["Liabilities"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "total_equity":        ["StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
                            "StockholdersEquity"],
    "liabilities_and_equity": ["LiabilitiesAndStockholdersEquity"],   # used to derive missing liabilities
    "revenue":             ["Revenues",
                            "RevenueFromContractWithCustomerExcludingAssessedTax",
                            "RevenueFromContractWithCustomerIncludingAssessedTax",
                            "SalesRevenueNet"],                        # older filings
    "net_income":          ["NetIncomeLoss", "ProfitLoss"],
    "operating_income":    ["OperatingIncomeLoss"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities",
                            "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
    #Oler featyres that are not used in the model but might be useful for analysis
    "liabilities_subject_to_compromise": ["LiabilitiesSubjectToCompromise"],
    "reorganization_items": ["ReorganizationItems"],
}
ALL_TAGS = [t for tags in FEATURE_TAGS.values() for t in tags]

Quarter directories: [PosixPath('/Users/pramukhvenkateshkoushik/Desktop/Data_cap/2025q1'), PosixPath('/Users/pramukhvenkateshkoushik/Desktop/Data_cap/2025q2'), PosixPath('/Users/pramukhvenkateshkoushik/Desktop/Data_cap/2025q3'), PosixPath('/Users/pramukhvenkateshkoushik/Desktop/Data_cap/2025q4'), PosixPath('/Users/pramukhvenkateshkoushik/Desktop/Data_cap/2026q1'), PosixPath('/Users/pramukhvenkateshkoushik/Desktop/Data_cap/2026q2')]


In [ ]:
# We choose iord and doc here to avoid reading the entire num.txt file, which is large. We only need the tag and iord columns to build our features.
# we choose custom =1 only to avoid tags that are not part of the standard taxonomy (e.g., custom tags added by a company).
def tag_read(q): 
    tag = pd.read_csv(q / "tag.txt", sep="\t", low_memory=False)
    return tag[(tag["custom"] == 0) & tag["tag"].isin(ALL_TAGS)]
tag = pd.concat([tag_read(q) for q in QUARTER_DIRS]).drop_duplicates("tag")
IORD = dict(zip(tag["tag"], tag["iord"]))                                          
tag[["tag", "iord", "doc"]]                                                         

,tag,iord,doc
73695,Assets,I,Sum of the carrying amounts as of the balance ...
73696,AssetsCurrent,I,Sum of the carrying amounts as of the balance ...
73720,Liabilities,I,Sum of the carrying amounts as of the balance ...
73721,LiabilitiesAndStockholdersEquity,I,"Amount of liabilities and equity items, includ..."
73722,LiabilitiesCurrent,I,Total obligations incurred as part of normal o...
73725,NetCashProvidedByUsedInOperatingActivities,D,Amount of cash inflow (outflow) from operating...
73726,NetIncomeLoss,D,"The portion of profit or loss for the period, ..."
73729,OperatingIncomeLoss,D,The net result for the period of deducting ope...
73756,Revenues,D,"Amount of revenue recognized from goods sold, ..."
73760,StockholdersEquity,I,Amount of equity (deficit) attributable to par...


In [ ]:
# We load a quarter here  and only choose specific columns required for us as num file is too big.
def load_quarter(qdir):
    sub = pd.read_csv(qdir / "sub.txt", sep="\t", low_memory=False,
                      usecols=["adsh", "cik", "name", "sic", "form", "period", "fy", "fp", "filed"])
    sub = sub[sub["form"].isin(FORMS)].dropna(subset=["period"])     
    sub["period"] = sub["period"].astype(int)                        

    num = pd.read_csv(qdir / "num.txt", sep="\t", low_memory=False)
    num = num[num["tag"].isin(ALL_TAGS)]                              
    # some of the earlier companies might not have segments column, so we add it if not present
    if "segments" not in num.columns:                                
        num["segments"] = np.nan
    return sub, num

In [ ]:
# We try to clean the num file and keep only the rows that are relevant for our analysis. We keep only rows with USD as the unit of measure,
# and we drop rows with missing values. We also merge with the sub file to keep only filings that are 10-K or 10-Q,
# and we keep only rows where the ddate matches the period of the filing. Finally, we add a column for iord based on the tag.
def clean_num(num, sub):
    num = num[num["segments"].isna() & num["coreg"].isna() & (num["uom"] == "USD")]
    num = num.dropna(subset=["value"])
    num = num.merge(sub[["adsh", "form", "period"]], on="adsh")       # also drops non-10-K/10-Q filings
    num = num[num["ddate"] == num["period"]].copy()                   # current period only
    num["iord"] = num["tag"].map(IORD)
    return num

In [ ]:
# Here we see qtrs being used to filter the data based on the filing type and the number of quarters. 
# For example, for 10-K filings, we keep only rows where qtrs is 0/4 (i.e., annual filings), while for 10-Q filings, we keep rows where 
# qtrs is between 1 and 3 (i.e., quarterly filings). We also drop duplicates based on adsh and tag to keep only the shortest window per 
# filing and tag.
def pick_period(num):
    annual = num["form"].str.startswith("10-K")
    keep = (((num["iord"] == "I") & (num["qtrs"] == 0)) |
            ((num["iord"] == "D") & annual & (num["qtrs"] == 4)) |
            ((num["iord"] == "D") & ~annual & num["qtrs"].between(1, 3)))
    num = num[keep].sort_values("qtrs")
    return num.drop_duplicates(["adsh", "tag"])                      

In [ ]:
# Here we build the features for the model. streamlining names for easy read of each companies
# we use _qtrs for duration of filing so we know if number 1-4 is used
def build_features(num):
    vals = num.pivot(index="adsh", columns="tag", values="value")
    qtrs = num.pivot(index="adsh", columns="tag", values="qtrs")
    out = pd.DataFrame(index=vals.index)
    for feature, tags in FEATURE_TAGS.items():
        present = [t for t in tags if t in vals.columns]
        if not present:
            out[feature] = np.nan
            continue
        out[feature] = vals[present].bfill(axis=1).iloc[:, 0]         
        if IORD.get(tags[0]) == "D":
            out[feature + "_qtrs"] = qtrs[present].bfill(axis=1).iloc[:, 0]
    return out.reset_index()

### Cell 7 – Fill missing total liabilities
**What:** Liabilities = (Liabilities + Equity) − Equity. **Why:** about 1 in 9 filings has no "Total liabilities"
line, but every balance sheet has the other two. A flag records which values are calculated.

In [28]:
def fill_liabilities(df):
    missing = df["total_liabilities"].isna()
    df["liabilities_derived"] = missing & df["liabilities_and_equity"].notna() & df["total_equity"].notna()
    df.loc[missing, "total_liabilities"] = df["liabilities_and_equity"] - df["total_equity"]
    return df

In [ ]:
#Do for all quarters so we can get latest filling version for each company as they might be filed mutplie times in a quarter. 
frames = []
for q in QUARTER_DIRS:
    sub, num = load_quarter(q)
    feats = fill_liabilities(build_features(pick_period(clean_num(num, sub))))
    frames.append(sub.merge(feats, on="adsh", how="inner").assign(quarter=q.name))
    print(f"filings from {q.name}")

df = pd.concat(frames, ignore_index=True)
df = df.sort_values("filed").drop_duplicates(["cik", "period"], keep="last")
df["period"] = pd.to_datetime(df["period"].astype(str), format="%Y%m%d")
df["filed"] = pd.to_datetime(df["filed"].astype(str), format="%Y%m%d")
df["xbrl_chapter11"] = df["liabilities_subject_to_compromise"].notna() | df["reorganization_items"].notna()
df = df.sort_values(["cik", "period"]).reset_index(drop=True)
df.head(20)

filings from 2025q1
filings from 2025q2
filings from 2025q3
filings from 2025q4
filings from 2026q1
filings from 2026q2


,adsh,cik,name,sic,form,period,fy,fp,filed,total_assets,...,operating_income,operating_income_qtrs,operating_cash_flow,operating_cash_flow_qtrs,liabilities_subject_to_compromise,reorganization_items,reorganization_items_qtrs,liabilities_derived,quarter,xbrl_chapter11
0,0001410578-25-000003,1750,AAR CORP,3720.0,10-Q,2024-11-30,2025.0,Q2,2025-01-08,2.849300e+09,...,NaN,NaN,3.400000e+06,2.0,NaN,NaN,NaN,True,2025q1,False
1,0001410578-25-000519,1750,AAR CORP,3720.0,10-Q,2025-02-28,2025.0,Q3,2025-03-28,2.859100e+09,...,NaN,NaN,-1.530000e+07,3.0,NaN,NaN,NaN,True,2025q1,False
2,0001410578-25-001475,1750,AAR CORP,3720.0,10-K,2025-05-31,2025.0,FY,2025-07-22,2.844600e+09,...,NaN,NaN,3.610000e+07,4.0,NaN,NaN,NaN,True,2025q3,False
3,0001104659-25-092589,1750,AAR CORP,3720.0,10-Q,2025-08-31,2026.0,Q1,2025-09-23,2.929700e+09,...,NaN,NaN,-4.490000e+07,1.0,NaN,NaN,NaN,True,2025q3,False
4,0001104659-26-001420,1750,AAR CORP,3720.0,10-Q,2025-11-30,2026.0,Q2,2026-01-07,3.242500e+09,...,NaN,NaN,-3.130000e+07,2.0,NaN,NaN,NaN,True,2026q1,False
5,0001104659-26-033973,1750,AAR CORP,3720.0,10-Q,2026-02-28,2026.0,Q3,2026-03-25,3.332500e+09,...,NaN,NaN,4.340000e+07,3.0,NaN,NaN,NaN,True,2026q1,False
6,0001628280-25-007110,1800,ABBOTT LABORATORIES,2834.0,10-K,2024-12-31,2024.0,FY,2025-02-21,8.141400e+10,...,6.825000e+09,4.0,8.558000e+09,4.0,NaN,NaN,NaN,True,2025q1,False
7,0001628280-25-021090,1800,ABBOTT LABORATORIES,2834.0,10-Q,2025-03-31,2025.0,Q1,2025-04-30,8.144800e+10,...,1.693000e+09,1.0,1.417000e+09,1.0,NaN,NaN,NaN,True,2025q2,False
8,0001628280-25-036720,1800,ABBOTT LABORATORIES,2834.0,10-Q,2025-06-30,2025.0,Q2,2025-07-30,8.399900e+10,...,2.052000e+09,1.0,3.464000e+09,2.0,NaN,NaN,NaN,True,2025q3,False
9,0001628280-25-047131,1800,ABBOTT LABORATORIES,2834.0,10-Q,2025-09-30,2025.0,Q3,2025-10-29,8.418100e+10,...,2.057000e+09,1.0,6.251000e+09,3.0,NaN,NaN,NaN,True,2025q4,False


In [ ]:
#Showing the percentage of non-missing values for each feature in the dataframe. Helps to understand our dataset completeness
features = list(FEATURE_TAGS)
print((df[features].notna().mean() * 100).round(1))
bs = df.dropna(subset=["total_assets", "total_liabilities", "total_equity"])

total_assets                         99.4
current_assets                       77.2
total_liabilities                    99.2
current_liabilities                  76.8
total_equity                         96.8
liabilities_and_equity               97.6
revenue                              73.6
net_income                           98.6
operating_income                     75.4
operating_cash_flow                  98.0
liabilities_subject_to_compromise     0.1
reorganization_items                  0.1
dtype: float64


In [ ]:
#Used AI here to get the URL specific for company from the SEC EDGAR database. This URL can be used to fetch the filings using speciifc CIK number.
import json, re, time, requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Pramukh_Venkatesh_Koushik koushik.p@northeastern.edu"}
CACHE = Path("/Users/pramukhvenkateshkoushik/Desktop/Data_cap/sec_cache")

def sec_get(url, name):
    path = CACHE / name
    if path.exists():
        return path.read_text(encoding="utf-8", errors="ignore")
    time.sleep(0.12)                                                  # stay under 10 requests/second
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        return None
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(r.text, encoding="utf-8")
    return r.text

In [ ]:
# Used AI help to get filings for each company stripping out only 8-K and 8-K/A filings. 
# These filings are used to identify bankruptcy events for the companies in our dataset.

events = []
for cik in df["cik"].unique():
    text = sec_get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json", f"sub/CIK{cik:010d}.json")
    if text is None:
        continue
    f = pd.DataFrame(json.loads(text)["filings"]["recent"])         
    if f.empty:
        continue
    is_bk = f["form"].isin(["8-K", "8-K/A"]) & f["items"].fillna("").str.contains(r"\b1\.03\b")
    events.append(f.loc[is_bk, ["accessionNumber", "filingDate", "primaryDocument"]].assign(cik=cik))

events = pd.concat(events, ignore_index=True) if events else \
    pd.DataFrame(columns=["accessionNumber", "filingDate", "primaryDocument", "cik"])
events["filingDate"] = pd.to_datetime(events["filingDate"])
events

,accessionNumber,filingDate,primaryDocument,cik
0,0001255294-09-000047,2009-02-13,mainbody.htm,13156
1,0000013156-08-000003,2008-07-18,mainbody.htm,13156
2,0001140361-20-019176,2020-08-27,brhc10014613_8k.htm,20520
3,0001140361-20-008876,2020-04-15,form8k.htm,20520
4,0001193125-23-187273,2023-07-14,d531800d8k.htm,28823
...,...,...,...,...
258,0001493152-26-039848,2026-08-24,form8-k.htm,1873722
259,0001193125-26-227832,2026-05-18,d130621d8k.htm,1901799
260,0001477932-25-004389,2025-06-03,icct_8k.htm,1906133
261,0001437749-24-034189,2024-11-08,blmh20241107c_8k.htm,1996210


In [ ]:
#Here using the links that i get previously , I get only the 1.03 items which states the bankruptcy filing for the company. 
# Used regex here to strip out 1.03. 
def item_103_text(row):
    acc = row["accessionNumber"].replace("-", "") # 
    url = f"https://www.sec.gov/Archives/edgar/data/{row['cik']}/{acc}/{row['primaryDocument']}"
    html = sec_get(url, f"docs/{acc}.htm")
    if html is None:
        return None
    text = re.sub(r"\s+", " ", BeautifulSoup(html, "html.parser").get_text(" "))
    
    m = re.search(r"((?:Item|ITEM)\s*1\.03.*?)(?=(?:Item|ITEM)\s*[2-9]\.\d\d\s+[A-Z]|SIGNATURE|$)", text, re.S)
    return (m.group(1) if m else text)[:4000]

events["item_103_text"] = events.apply(item_103_text, axis=1) if len(events) else None
events.to_csv("/Users/pramukhvenkateshkoushik/Desktop/Data_cap/bankruptcy_events_8k.csv", index=False)               

In [ ]:
# Tried to find the earliest bankruptcy event for each filing
pairs = df[["adsh", "cik", "filed"]].merge(events, on="cik")
gap = (pairs["filingDate"] - pairs["filed"]).dt.days

future = pairs[(gap > 0) & (gap <= 365)].sort_values("filingDate").drop_duplicates("adsh")
future = future.rename(columns={"accessionNumber": "bk_8k_adsh", "filingDate": "bk_8k_date"})
df = df.merge(future[["adsh", "bk_8k_adsh", "bk_8k_date", "item_103_text"]], on="adsh", how="left")

df["bankrupt_12m"] = df["bk_8k_adsh"].notna().astype(float)
df.loc[(df["filed"] + pd.Timedelta(days=365) > pd.Timestamp.today()) & (df["bankrupt_12m"] == 0),
       "bankrupt_12m"] = np.nan
df["prior_bankruptcy"] = df["adsh"].isin(pairs.loc[(gap <= 0) & (gap > -730), "adsh"])

In [ ]:
#Made tables spceifically for different years and then use for model training and investigator agent as text dont go to the model  
df1_y = df[df['quarter'].str.startswith('2025')]
df2_y = df[df['quarter'].str[:4].isin(['2025','2026'])]
for name,d in [('2025',df1_y), ('2025_2026',df2_y)]:
    print(name,d.shape,d["bankrupt_12m"].value_counts(dropna=False).to_dict())
    d.to_csv(f"/Users/pramukhvenkateshkoushik/Desktop/Data_cap/final_dataset_{name}.csv", index=False)
    d.drop(columns=["name", "bk_8k_adsh", "bk_8k_date", "item_103_text"]).to_csv(
        f"/Users/pramukhvenkateshkoushik/Desktop/Data_cap/model_ready_{name}.csv", index=False)

2025 (22344, 34) {0.0: 16615, nan: 5591, 1.0: 138}
2025_2026 (33506, 34) {nan: 16710, 0.0: 16615, 1.0: 181}


In [28]:
final_df = pd.read_csv("/Users/pramukhvenkateshkoushik/Desktop/Data_cap/final_dataset_2025_2026.csv")

/var/folders/yb/5s1v80557r3cv2sty2xp2g_00000gn/T/ipykernel_43592/4073413931.py:1: DtypeWarning: Columns (29,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv("/Users/pramukhvenkateshkoushik/Desktop/Data_cap/final_dataset_2025_2026.csv")


In [29]:
final_df.isna().mean()

adsh                                 0.000000
cik                                  0.000000
name                                 0.000000
sic                                  0.031397
form                                 0.000000
period                               0.000000
fy                                   0.000030
fp                                   0.000030
filed                                0.000000
total_assets                         0.006208
current_assets                       0.228108
total_liabilities                    0.007521
current_liabilities                  0.231929
total_equity                         0.031785
liabilities_and_equity               0.023727
revenue                              0.264281
revenue_qtrs                         0.264281
net_income                           0.013669
net_income_qtrs                      0.013669
operating_income                     0.245777
operating_income_qtrs                0.245777
operating_cash_flow               

In [30]:
final_df[['revenue','current_assets','operating_income']].isna().groupby(final_df['sic']//1000,dropna=False).mean()

,revenue,current_assets,operating_income
sic,,,
0.0,0.102439,0.043902,0.121951
1.0,0.198188,0.073613,0.174972
2.0,0.296350,0.010543,0.063747
3.0,0.060616,0.014649,0.064489
4.0,0.116338,0.008093,0.057157
5.0,0.070167,0.023062,0.081452
6.0,0.520606,0.709170,0.626512
7.0,0.067046,0.038782,0.083699
8.0,0.102884,0.023383,0.079501


In [31]:
for col in ['fp','form']:
    print(final_df[col].value_counts(dropna=False))

fp
Q1     10538
FY     10401
Q3      6305
Q2      6261
NaN        1
Name: count, dtype: int64
form
10-Q      22820
10-K      10091
10-K/A      311
10-Q/A      284
Name: count, dtype: int64


In [32]:
print(pd.crosstab(final_df['fp'],final_df['revenue_qtrs'])) 
print(pd.crosstab(final_df['fp'],final_df['operating_cash_flow_qtrs']))

revenue_qtrs   1.0  2.0  3.0   4.0
fp                                
FY               0    0    0  7692
Q1            7589    0    0     0
Q2            4624   37    0     0
Q3            4660    1   47     0
operating_cash_flow_qtrs    1.0   2.0   3.0    4.0
fp                                                
FY                            0     0     0  10099
Q1                        10354     0     0      0
Q2                           96  6082     1      0
Q3                          100    25  6085      0


In [ ]:
print(final_df.shape)

(33506, 34)


In [34]:
events = final_df[final_df["bk_8k_adsh"].notna()]
events = events[["cik", "name", "bk_8k_adsh", "bk_8k_date", "item_103_text"]]
events = events.drop_duplicates(["cik", "bk_8k_adsh"])
print(len(events), "bankruptcy 8-Ks")

64 bankruptcy 8-Ks


In [43]:
text = events["item_103_text"].fillna("").str.lower()
text = text.str.replace(r"\s+", " ", regex=True)                 # collapse odd spaces
text = text.str.replace("item 1.03.", "item 1.03", regex=False)  # "Item 

In [44]:
text

2559     item 1.03 of this current report on form 8-k i...
3622     item 1.03 bankruptcy or receivership. on june ...
3650     item 1.03 bankruptcy or receivership. on janua...
3801     item 1.03 of this current report on form 8-k (...
4325     item 1.03 bankruptcy or receivership. on march...
                               ...                        
28734    item 1.03 bankruptcy or receivership. on march...
29044     false 0001873722 0001873722 2026-08-18 2026-0...
29782    item 1.03 bankruptcy or receivership. voluntar...
29844    item 1.03 bankruptcy or receivership. on june ...
31676    item 1.03 bankruptcy or receivership. on janua...
Name: item_103_text, Length: 64, dtype: object